# Train Route Analysis and Journey Time Prediction

## Level 1: Data Overview

### Project Description

This project focuses on analyzing train journey data and developing
a system for predicting train journey duration.

The project follows a structured data science workflow that includes
data understanding, data cleaning, feature engineering, exploratory
data analysis, visualization, and machine learning.

### Level 1: Data Overview

The purpose of this level is to understand the structure, completeness,
and basic characteristics of the train dataset before performing
data cleaning and further analysis.

### Objectives of Level 1

- Create a summary of the dataset
- Understand the available columns and data types
- Prepare a train-wise table showing route start and end stations
- Calculate basic statistics for distance and number of stops
- Identify missing, null, duplicate, or potentially inconsistent values

In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("../data/Internship Dataset.csv")

In [6]:
df.head()

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance
0,1,107,SWV,100,100,100,100,SAWANTWADI R,1,00:00:00,10:25:00,0
1,2,107,THVM,260,228,196,164,THIVIM,1,11:06:00,11:08:00,32
2,3,107,KRMI,345,296,247,198,KARMALI,1,11:28:00,11:30:00,49
3,4,107,MAO,490,412,334,256,MADGOAN JN.,1,12:10:00,00:00:00,78
4,1,108,MAO,100,100,100,100,MADGOAN JN.,1,00:00:00,20:30:00,0


In [7]:
df.shape

(186074, 12)

In [8]:
print("Total records:", df.shape[0])
print("Total columns:", df.shape[1])

Total records: 186074
Total columns: 12


In [9]:
df.columns

Index(['SN', 'Train_No', 'Station_Code', '1A', '2A', '3A', 'SL',
       'Station_Name', 'Route_Number', 'Arrival_time', 'Departure_Time',
       'Distance'],
      dtype='str')

In [10]:
print("Column names:")
for column in df.columns:
    print("-", column)

Column names:
- SN
- Train_No
- Station_Code
- 1A
- 2A
- 3A
- SL
- Station_Name
- Route_Number
- Arrival_time
- Departure_Time
- Distance


In [11]:
print("Data types of each column:")
print(df.dtypes)

Data types of each column:
SN                int64
Train_No          int64
Station_Code        str
1A                int64
2A                int64
3A                int64
SL                int64
Station_Name        str
Route_Number      int64
Arrival_time        str
Departure_Time      str
Distance          int64
dtype: object


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 186074 entries, 0 to 186073
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   SN              186074 non-null  int64
 1   Train_No        186074 non-null  int64
 2   Station_Code    186074 non-null  str  
 3   1A              186074 non-null  int64
 4   2A              186074 non-null  int64
 5   3A              186074 non-null  int64
 6   SL              186074 non-null  int64
 7   Station_Name    186074 non-null  str  
 8   Route_Number    186074 non-null  int64
 9   Arrival_time    186074 non-null  str  
 10  Departure_Time  186074 non-null  str  
 11  Distance        186074 non-null  int64
dtypes: int64(8), str(4)
memory usage: 17.0 MB


In [13]:
unique_trains = df["Train_No"].nunique()

print("Total unique trains:", unique_trains)

Total unique trains: 11113


In [14]:
unique_stations = df["Station_Code"].nunique()

print("Total unique stations:", unique_stations)

Total unique stations: 8147


In [15]:
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

Missing values in each column:
SN                0
Train_No          0
Station_Code      0
1A                0
2A                0
3A                0
SL                0
Station_Name      0
Route_Number      0
Arrival_time      0
Departure_Time    0
Distance          0
dtype: int64


In [16]:
duplicate_rows = df.duplicated().sum()

print("Total duplicate rows:", duplicate_rows)

Total duplicate rows: 0


In [17]:
df["Distance"].describe()

count    186074.000000
mean        281.353838
std         483.743964
min           0.000000
25%          23.000000
50%          73.000000
75%         291.000000
max        4260.000000
Name: Distance, dtype: float64

In [18]:
stations_per_train = df.groupby("Train_No").size()

stations_per_train.describe()

count    11113.000000
mean        16.743814
std         12.993123
min          2.000000
25%          8.000000
50%         15.000000
75%         22.000000
max        118.000000
dtype: float64

In [19]:
df[df["Train_No"] == df["Train_No"].iloc[0]][
    ["Train_No", "Station_Code", "Station_Name", "Arrival_time", "Departure_Time", "Distance"]
]

,Train_No,Station_Code,Station_Name,Arrival_time,Departure_Time,Distance
0,107,SWV,SAWANTWADI R,00:00:00,10:25:00,0
1,107,THVM,THIVIM,11:06:00,11:08:00,32
2,107,KRMI,KARMALI,11:28:00,11:30:00,49
3,107,MAO,MADGOAN JN.,12:10:00,00:00:00,78


In [20]:
train_routes = df.groupby("Train_No").agg(
    Start_Station=("Station_Name", "first"),
    End_Station=("Station_Name", "last")
).reset_index()

train_routes.head(10)

,Train_No,Start_Station,End_Station
0,107,SAWANTWADI R,MADGOAN JN.
1,108,MADGOAN JN.,SAWANTWADI R
2,128,MADGOAN JN.,CHHATRAPATI
3,290,DELHI-SAFDAR,DELHI-SAFDAR
4,401,AURANGABAD,VARANASI JN.
5,421,LUCKNOW JN.,SHRI MATA VA
6,422,SHRI MATA VA,LUCKNOW JN.
7,477,SIRSA,SIRSA
8,502,RAJENDRANAGA,AMBALA CANTT
9,504,PATNA JN.,BATHINDA JN


In [21]:
zero_distance = (df["Distance"] == 0).sum()

print("Records with Distance = 0:", zero_distance)

Records with Distance = 0: 11124


In [22]:
zero_distance_rows = df[df["Distance"] == 0]

zero_distance_rows[[
    "Train_No",
    "Station_Code",
    "Station_Name",
    "Route_Number",
    "Distance"
]].head(20)

,Train_No,Station_Code,Station_Name,Route_Number,Distance
0,107,SWV,SAWANTWADI R,1,0
4,108,MAO,MADGOAN JN.,1,0
8,128,MAO,MADGOAN JN.,1,0
30,290,DSJ,DELHI-SAFDAR,1,0
44,401,AWB,AURANGABAD,1,0
56,421,LKO,LUCKNOW JN.,1,0
61,422,SVDK,SHRI MATA VA,1,0
66,477,SSA,SIRSA,1,0
80,502,RJPB,RAJENDRANAGA,1,0
89,504,PNBE,PATNA JN.,1,0


In [23]:
route_counts = df.groupby("Train_No")["Route_Number"].nunique()

print("Maximum different Route_Number values for one train:",
      route_counts.max())

print("\nTrains with more than one Route_Number:")
print(route_counts[route_counts > 1].head(20))

Maximum different Route_Number values for one train: 1

Trains with more than one Route_Number:
Series([], Name: Route_Number, dtype: int64)


In [24]:
sn_order_check = df.groupby("Train_No")["SN"].apply(
    lambda x: x.is_monotonic_increasing
)

print("Trains with correctly ordered SN values:",
      sn_order_check.sum())

print("Total trains:", len(sn_order_check))

Trains with correctly ordered SN values: 11113
Total trains: 11113


In [25]:
negative_distance = (df["Distance"] < 0).sum()

print("Records with negative Distance:", negative_distance)

Records with negative Distance: 0
